# 02 · Qualidade do bronze — pedidos e entregas

Segunda varredura de qualidade: o coração operacional (pedidos, itens, linha do tempo de fases, entregas, ocorrências, minutas e notas fiscais). Mesmo método do notebook 01: a pergunta em **SQL (DuckDB)**, o resultado em **pandas**, a **Nota Técnica** fechando cada seção; nenhum número digitado.

**Como rodar:** com o bronze extraído (`uv run bronze-staging`), kernel da `.venv` do projeto.

In [1]:
import pandas as pd

from logistica_fictitur.lake import conectar

pd.set_option("display.max_rows", 60)
conn = conectar()


def sql(consulta: str) -> pd.DataFrame:
    return conn.execute(consulta).df()

## 1. Panorama e janela temporal

Os volumes do domínio e o intervalo de tempo que a operação cobre.

In [2]:
sql("""
    SELECT 'pedido' AS tabela, count(*) AS linhas,
           min(dt_solicitacao)::date AS inicio, max(dt_solicitacao)::date AS fim
    FROM bronze_fwm_pedido
    UNION ALL SELECT 'pedido_item', count(*), NULL, NULL FROM bronze_fwm_pedido_item
    UNION ALL SELECT 'pedido_fase', count(*),
           min(dt_entrada)::date, max(dt_entrada)::date FROM bronze_fwm_pedido_fase
    UNION ALL SELECT 'entrega', count(*),
           min(dt_entrega)::date, max(dt_entrega)::date FROM bronze_expedicao_entrega
    UNION ALL SELECT 'ocorrencia_entrega', count(*), NULL, NULL
    FROM bronze_expedicao_ocorrencia_entrega
    UNION ALL SELECT 'minuta', count(*), NULL, NULL FROM bronze_expedicao_minuta
    UNION ALL SELECT 'nf', count(*), NULL, NULL FROM bronze_fwm_nf
    ORDER BY linhas DESC
""")

,tabela,linhas,inicio,fim
0,pedido_fase,8355683,1899-12-30,2098-08-22
1,pedido_item,4515355,NaT,NaT
2,pedido,1033263,2020-01-01,2026-09-01
3,nf,1030210,NaT,NaT
4,entrega,1030210,1899-12-30,2098-08-22
5,ocorrencia_entrega,85052,NaT,NaT
6,minuta,41228,NaT,NaT


> **Nota Técnica 1** · **Observado:** os volumes são coerentes entre si (itens por pedido na razão esperada, uma entrega e uma NF por pedido concluído), mas a **janela temporal da entrega denuncia o problema da seção 2**: as datas extremas saltam para fora do intervalo de operação da empresa. · **Por que importa:** o `min`/`max` de uma coluna de data é o teste de fumaça mais barato que existe, e aqui ele já acusou. · **Ação proposta:** quantificar e classificar as datas impossíveis antes de qualquer análise de prazo.

## 2. Datas-sentinela e impossíveis na entrega

Sistemas antigos usam datas-mágicas (o famoso `1899-12-30` de heranças VB/Access) e erros de digitação produzem futuros absurdos. Aqui: qualquer entrega fora da vida útil plausível da operação.

In [3]:
sql("""
    SELECT dt_entrega::date AS data_impossivel, count(*) AS entregas
    FROM bronze_expedicao_entrega
    WHERE dt_entrega IS NOT NULL
      AND dt_entrega::date NOT BETWEEN DATE '2019-01-01' AND DATE '2027-12-31'
    GROUP BY 1 ORDER BY 2 DESC
""")

,data_impossivel,entregas
0,1899-12-30,1742
1,2055-08-11,1706
2,2098-08-22,1678


In [4]:
sql("""
    SELECT count(*) AS entregas,
           count(*) FILTER (WHERE dt_entrega::date
               NOT BETWEEN DATE '2019-01-01' AND DATE '2027-12-31') AS impossiveis,
           round(100.0 * count(*) FILTER (WHERE dt_entrega::date
               NOT BETWEEN DATE '2019-01-01' AND DATE '2027-12-31') / count(*), 3)
               AS pct
    FROM bronze_expedicao_entrega WHERE dt_entrega IS NOT NULL
""")

,entregas,impossiveis,pct
0,1030210,5126,0.498


> **Nota Técnica 2** · **Observado:** um conjunto pequeno em percentual, mas de milhares de linhas, com três datas impossíveis recorrentes: a sentinela clássica de 1899 e dois futuros absurdos (2055 e 2098), repetidos demais para serem digitação aleatória: são valores-mágicos de sistema. · **Por que importa:** essas linhas **contaminam qualquer métrica de prazo**: a de 1899 parece entrega antecipadíssima, as futuras parecem atrasos de décadas; um OTIF calculado sem excluí-las mente para os dois lados. · **Ação proposta:** a silver **anula a data impossível e marca** (`fl_data_entrega_invalida`), preservando o valor original em coluna de auditoria; não se adivinha a data verdadeira (ausência não se preenche).

## 3. A linha do tempo de fases está em ordem?

Cada pedido percorre fases com ordem canônica (EA → ... → EC). A janela abaixo usa **window function** (`LAG` particionado por pedido) para achar fases registradas **antes** da fase anterior, o teste de sanidade de qualquer timeline.

In [5]:
sql("""
    WITH linha_do_tempo AS (
        SELECT pf.ss, f.codigo, f.ordem, pf.dt_entrada,
               lag(pf.dt_entrada) OVER (PARTITION BY pf.ss ORDER BY f.ordem)
                   AS entrada_anterior
        FROM bronze_fwm_pedido_fase pf
        JOIN bronze_cadastro_fase f ON f.id = pf.fase_id
    )
    SELECT codigo AS fase_fora_de_ordem,
           count(*) AS ocorrencias,
           count(DISTINCT ss) AS pedidos_afetados
    FROM linha_do_tempo
    WHERE dt_entrada < entrada_anterior
    GROUP BY 1 ORDER BY 2 DESC
""")

,fase_fora_de_ordem,ocorrencias,pedidos_afetados
0,EC,26262,26262


In [6]:
sql("""
    WITH fora_de_ordem AS (
        SELECT DISTINCT pf.ss
        FROM bronze_fwm_pedido_fase pf
        JOIN bronze_cadastro_fase f ON f.id = pf.fase_id
        QUALIFY pf.dt_entrada < lag(pf.dt_entrada)
            OVER (PARTITION BY pf.ss ORDER BY f.ordem)
    )
    SELECT count(*) AS pedidos_fora_de_ordem,
           count(*) FILTER (WHERE e.dt_entrega::date
               NOT BETWEEN DATE '2019-01-01' AND DATE '2027-12-31')
               AS explicados_por_data_impossivel,
           count(*) FILTER (WHERE e.dt_entrega::date
               BETWEEN DATE '2019-01-01' AND DATE '2027-12-31')
               AS com_data_plausivel
    FROM fora_de_ordem fo
    JOIN bronze_expedicao_entrega e USING (ss)
""")

,pedidos_fora_de_ordem,explicados_por_data_impossivel,com_data_plausivel
0,26262,1742,24520


> **Nota Técnica 3** · **Observado:** as violações de ordem se concentram na fase final (EC, a entrega): parte é arrasto direto das datas impossíveis da seção 2, e parte tem data plausível porém **anterior às fases internas**: a entrega registrada antes da emissão, o clássico do apontamento feito depois do fato com data retroativa. · **Por que importa:** durações por fase (a matéria-prima da análise de gargalo) ficam negativas ou absurdas nesses pedidos; já vimos essa doença na referência do nosso universo, e a lição é a mesma. · **Ação proposta:** a silver marca o pedido (`fl_timeline_inconsistente`) sem reordenar nada: timeline não se conserta por palpite, e o marcado sai das apurações de duração.

## 4. Prazos e motivos de atraso

Contratos de completude condicional: entrega precisa de prazo; atraso pede motivo; pontualidade não deveria carregar motivo.

In [7]:
sql("""
    WITH plausiveis AS (
        SELECT * FROM bronze_expedicao_entrega
        WHERE dt_entrega IS NOT NULL
          AND dt_entrega::date BETWEEN DATE '2019-01-01' AND DATE '2027-12-31'
    )
    SELECT 'entrega sem prazo prometido' AS verificacao,
           count(*) FILTER (WHERE prazo_inicial_cliente IS NULL) AS ocorrencias
    FROM plausiveis
    UNION ALL
    SELECT 'atrasada sem motivo de atraso',
           count(*) FILTER (WHERE dt_entrega::date > prazo_inicial_cliente
                            AND motivo_atraso_id IS NULL)
    FROM plausiveis
    UNION ALL
    SELECT 'no prazo carregando motivo de atraso',
           count(*) FILTER (WHERE dt_entrega::date <= prazo_inicial_cliente
                            AND motivo_atraso_id IS NOT NULL)
    FROM plausiveis
    UNION ALL
    SELECT 'data impossivel travestida de atraso (sem motivo)',
           count(*) FILTER (WHERE dt_entrega::date > DATE '2027-12-31'
                            AND motivo_atraso_id IS NULL)
    FROM bronze_expedicao_entrega WHERE dt_entrega IS NOT NULL
""")

,verificacao,ocorrencias
0,entrega sem prazo prometido,0
1,atrasada sem motivo de atraso,0
2,no prazo carregando motivo de atraso,0
3,data impossivel travestida de atraso (sem motivo),2996


> **Nota Técnica 4** · **Observado:** dentro do universo de datas plausíveis, os contratos condicionais fecham (toda atrasada tem motivo, nenhuma pontual carrega motivo, prazo sempre presente); as exceções aparentes são as linhas de data impossível, que **se disfarçam de atraso sem motivo**. · **Por que importa:** é a demonstração prática de por que a sentinela se trata primeiro: sem a marcação da seção 2, um relatório de "atrasos sem justificativa" acusaria a operação por culpa do dado. · **Ação proposta:** nenhuma regra nova; reforça a precedência da regra da seção 2 na ordem de conformação da silver.

## 5. Consistência entre as pontas do fluxo

O amarrado do fluxo: pedido concluído tem entrega; entrega tem NF; carga rodou em minuta quando devia (e retirada, nunca).

In [8]:
sql("""
    SELECT 'pedido em EC sem registro de entrega' AS verificacao, count(*) AS ocorrencias
    FROM bronze_fwm_pedido p
    JOIN bronze_cadastro_fase f ON f.id = p.fase_id AND f.codigo = 'EC'
    LEFT JOIN bronze_expedicao_entrega e USING (ss)
    WHERE e.ss IS NULL
    UNION ALL
    SELECT 'entrega sem nota fiscal', count(*)
    FROM bronze_expedicao_entrega e
    LEFT JOIN bronze_fwm_nf n USING (ss)
    WHERE n.ss IS NULL
    UNION ALL
    SELECT 'entregue nao-retira sem minuta', count(*)
    FROM bronze_expedicao_entrega e
    JOIN bronze_fwm_pedido p USING (ss)
    JOIN bronze_cadastro_modalidade m ON m.id = p.modalidade_id
    LEFT JOIN bronze_expedicao_minuta_pedido mp USING (ss)
    WHERE m.codigo <> 'RETIRA' AND mp.ss IS NULL
    UNION ALL
    SELECT 'retirada rodando em minuta (nao deveria)', count(*)
    FROM bronze_expedicao_minuta_pedido mp
    JOIN bronze_fwm_pedido p USING (ss)
    JOIN bronze_cadastro_modalidade m ON m.id = p.modalidade_id
    WHERE m.codigo = 'RETIRA'
""")

,verificacao,ocorrencias
0,pedido em EC sem registro de entrega,0
1,entrega sem nota fiscal,0
2,entregue nao-retira sem minuta,0
3,retirada rodando em minuta (nao deveria),0


> **Nota Técnica 5** · **Observado:** o fluxo amarra ponta a ponta: todo pedido concluído tem entrega, toda entrega tem NF, toda carga entregue rodou em minuta e nenhuma retirada aparece em romaneio. · **Por que importa:** essas amarras são as junções que a gold vai usar milhares de vezes; confirmá-las aqui evita surpresa lá. · **Ação proposta:** checagens promovidas à régua permanente, sem correção.

## 6. Consolidação dos achados do domínio

Os achados de pedidos e entregas, recalculados em célula, para o catálogo da silver.

In [9]:
achados = sql("""
    SELECT 'PED-01' AS codigo,
           'entregas com data impossivel (sentinelas e futuros)' AS achado,
           (SELECT count(*) FROM bronze_expedicao_entrega
            WHERE dt_entrega IS NOT NULL AND dt_entrega::date
                NOT BETWEEN DATE '2019-01-01' AND DATE '2027-12-31') AS medida,
           'anular data + flag + valor original em auditoria' AS acao_proposta
    UNION ALL
    SELECT 'PED-02', 'pedidos com linha do tempo de fases fora de ordem',
           (SELECT count(*) FROM (
               SELECT DISTINCT pf.ss FROM bronze_fwm_pedido_fase pf
               JOIN bronze_cadastro_fase f ON f.id = pf.fase_id
               QUALIFY pf.dt_entrada < lag(pf.dt_entrada)
                   OVER (PARTITION BY pf.ss ORDER BY f.ordem))),
           'flag de timeline inconsistente; nao reordenar'
    UNION ALL
    SELECT 'PED-03', 'timeline fora de ordem COM data de entrega plausivel',
           (SELECT count(*) FROM (
               SELECT DISTINCT pf.ss FROM bronze_fwm_pedido_fase pf
               JOIN bronze_cadastro_fase f ON f.id = pf.fase_id
               QUALIFY pf.dt_entrada < lag(pf.dt_entrada)
                   OVER (PARTITION BY pf.ss ORDER BY f.ordem)) fo
            JOIN bronze_expedicao_entrega e USING (ss)
            WHERE e.dt_entrega::date BETWEEN DATE '2019-01-01' AND DATE '2027-12-31'),
           'mesma flag; excluir das apuracoes de duracao por fase'
""")
achados

,codigo,achado,medida,acao_proposta
0,PED-01,entregas com data impossivel (sentinelas e fut...,5126,anular data + flag + valor original em auditoria
1,PED-02,pedidos com linha do tempo de fases fora de ordem,26262,flag de timeline inconsistente; nao reordenar
2,PED-03,timeline fora de ordem COM data de entrega pla...,24520,mesma flag; excluir das apuracoes de duracao p...


> **Nota Técnica 6 (fecho)** · **Observado:** dois problemas-raiz no domínio (datas impossíveis na entrega e timeline de fases fora de ordem), sendo que o segundo tem um recorte que sobrevive mesmo com data plausível: o apontamento retroativo. Prazos, motivos e amarras do fluxo estão íntegros quando as datas impossíveis são postas de lado. · **Por que importa:** os dois achados atacam exatamente as métricas que o BI existe para servir (OTIF e duração por fase); tratá-los é pré-condição de qualquer número confiável. · **Ação proposta:** PED-01..03 vão ao catálogo da silver. Próxima varredura: **03 · estoque e financeiro** (validades, saldos e o furo fiscal encontrando as margens).